# Credit Value-at-Risk — Homework 5
**Group 15:** Pietro De Bernardi (69791), Giammarco Ricciardulli (75040), Lucas Laussegger (73038)

Credit Risk 2025/26

## Setup

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import norm, multivariate_normal
from scipy.stats import beta as beta_dist

np.random.seed(42)

DATA_FILE  = "CreditVaR_data.xlsx"
RATINGS    = ["AAA", "AA", "A", "BBB", "BB", "B", "CCC", "Default"]
IR_RATINGS = RATINGS[:-1]
N_SIM      = 100_000
CONF_LEVEL = 0.01

# Factor correlation matrix (T, L, C)
FACTOR_CORR = np.array([[1.0, 0.3, 0.1],
                         [0.3, 1.0, 0.2],
                         [0.1, 0.2, 1.0]])

## Load and clean data

In [2]:
# Interest rates: rows = rating categories, columns = year 1-4
ir = pd.read_excel(DATA_FILE, sheet_name="interest", index_col=0, header=None, skiprows=2)
ir = ir.dropna(how="all").astype(float)
ir.index   = IR_RATINGS
ir.columns = [1, 2, 3, 4]

# Transition matrix
tr = pd.read_excel(DATA_FILE, sheet_name="transition", index_col=0, header=None, skiprows=2)
tr = tr.dropna(how="all").astype(float)
tr.index   = IR_RATINGS
tr.columns = RATINGS

# Bonds portfolio
bonds = pd.read_excel(DATA_FILE, sheet_name="bonds", index_col=0)
bonds.columns = ["CR", "Maturity", "Rating", "R_mean", "R_std", "Beta_T", "Beta_L", "Beta_C"]

# Discount factor table
DFS = {
    r: [1.0] + [1.0 / (1 + ir.loc[r, t] / 100) ** t for t in [1, 2, 3, 4]]
    for r in IR_RATINGS
}

print(ir)
print(bonds)

         1      2      3      4
AAA   3.60   4.17   4.73   5.12
AA    3.65   4.22   4.78   5.17
A     3.72   4.32   4.93   5.32
BBB   4.10   4.67   5.25   5.63
BB    5.55   6.02   6.78   7.27
B     6.05   7.02   8.03   8.52
CCC  15.05  15.02  14.03  13.52
          CR  Maturity  Rating  R_mean  R_std  Beta_T  Beta_L  Beta_C
issue_id                                                             
1          6         5       4   51.13     26     0.4     0.2     0.0
2          5         3       3   51.13     26     0.0     0.0     0.9
3          5         2       2   56.00     24     0.5     0.3     0.0
4          3         1       1   56.00     24     0.1     0.4     0.3
5          5         5       3   56.00     24     0.5     0.2     0.1
6          8         3       6   35.00     22     0.0     0.9     0.0
7          8         1       6   35.00     25     0.4     0.2     0.0
8          9         2       7   35.00     22     0.6     0.1     0.0
9          6         2       4   42.00     2

## Core functions

In [3]:
def bond_price_vector(maturity: int, coupon: float, recovery: float) -> list:
    """
    Return a list of 8 bond prices — one per destination rating class
    (7 non-default + 1 default at the recovery value).
    """
    prices = []
    for dest in IR_RATINGS:
        dfs = DFS[dest][:maturity]
        cfs = [coupon] * maturity
        cfs[-1] += 100
        prices.append(sum(cf * df for cf, df in zip(cfs, dfs)))
    prices.append(recovery)
    return prices


def z_thresholds(rating: str) -> list:
    """
    Compute z-score rating-change thresholds for a given initial rating.
    Returns a 9-element list: [+inf, z_AAA, z_AA, ..., z_CCC, sentinel].
    z_k is the lower boundary of rating class k: r > z_k implies migration
    to rating k or better.
    """
    probs      = tr.loc[rating].values / 100
    cumulative = np.cumsum(probs[::-1])[::-1]
    return list(norm.ppf(cumulative)) + [-1e9]


def joint_prob_matrix(thr1: list, thr2: list, corr: float) -> np.ndarray:
    """
    Build an 8x8 joint transition probability matrix for two bonds
    whose standardised returns have correlation corr.
    """
    mvn = multivariate_normal(mean=[0, 0], cov=[[1, corr], [corr, 1]])
    n   = len(thr1) - 1
    mat = np.zeros((n, n))
    for i in range(n):
        for k in range(n):
            mat[i, k] = (mvn.cdf([thr1[i],   thr2[k]])
                       - mvn.cdf([thr1[i+1], thr2[k]])
                       - mvn.cdf([thr1[i],   thr2[k+1]])
                       + mvn.cdf([thr1[i+1], thr2[k+1]]))
    return mat


def sim_bond_prices(r_sim: np.ndarray, thr: list, price_vec: list) -> np.ndarray:
    """
    Vectorised rating-to-price mapping.
    Iterates thresholds from CCC to AAA so each observation is assigned
    the price of the highest rating threshold it exceeds.
    """
    thr_inner = thr[1:-1]
    out = np.full(len(r_sim), price_vec[-1], dtype=float)
    for i in range(len(thr_inner) - 1, -1, -1):
        out = np.where(r_sim > thr_inner[i], price_vec[i], out)
    return out

## Warm-up (not graded)

In [4]:
thr_BBB  = z_thresholds("BBB")
thr_A    = z_thresholds("A")
prices_1 = bond_price_vector(5, 6,  51.13)
prices_2 = bond_price_vector(3, 5,  51.13)
prob_BBB = tr.loc["BBB"].tolist()
prob_A   = tr.loc["A"].tolist()

jp_wu      = joint_prob_matrix(thr_BBB, thr_A, corr=0.30)
combined   = np.add.outer(prices_1, prices_2).flatten()
probs_flat = jp_wu.flatten()
order      = np.argsort(combined)
csum       = np.cumsum(probs_flat[order])
idx_wu     = next(i for i in range(len(csum) - 1)
                  if csum[i] < CONF_LEVEL <= csum[i + 1])

v_star_wu = combined[order][idx_wu]
mean_wu   = (sum(v * p / 100 for v, p in zip(prices_1, prob_BBB))
           + sum(v * p / 100 for v, p in zip(prices_2, prob_A)))

print(f"Warm-up (rho = 0.30)")
print(f"  muV        = {mean_wu:.2f}")
print(f"  V*         = {v_star_wu:.2f}")
print(f"  Credit VaR = {abs(v_star_wu - mean_wu):.2f}")

Warm-up (rho = 0.30)
  muV        = 213.27
  V*         = 203.73
  Credit VaR = 9.54


---
## Exercise 1 — Analytical Credit VaR (ρ = 0.072)

Same two-bond portfolio; correlation ρ = 0.072 derived from the factor structure
of Example 4.1.2. Analytical CreditMetrics method — no simulation required.

In [5]:
CORR_EX1  = 0.072

jp_ex1    = joint_prob_matrix(thr_BBB, thr_A, corr=CORR_EX1)
combined  = np.add.outer(prices_1, prices_2).flatten()
probs_ex1 = jp_ex1.flatten()
order_ex1 = np.argsort(combined)
csum_ex1  = np.cumsum(probs_ex1[order_ex1])
idx_ex1   = next(i for i in range(len(csum_ex1) - 1)
                 if csum_ex1[i] < CONF_LEVEL <= csum_ex1[i + 1])

v_star_1 = combined[order_ex1][idx_ex1]
mean_ex1 = (sum(v * p / 100 for v, p in zip(prices_1, prob_BBB))
          + sum(v * p / 100 for v, p in zip(prices_2, prob_A)))
cvar_ex1 = abs(v_star_1 - mean_ex1)

print(f"Exercise 1  (rho = {CORR_EX1})")
print(f"  muV        = {mean_ex1:.4f}")
print(f"  V*         = {v_star_1:.4f}")
print(f"  Credit VaR = {cvar_ex1:.4f}")

Exercise 1  (rho = 0.072)
  muV        = 213.2708
  V*         = 203.7286
  Credit VaR = 9.5423


---
## Exercise 2 — Simulation-based Credit VaR (2 bonds)

Bond 1 (BBB) = Vivendi:  r_Viv = 0.4 T + 0.2 L + eps_Viv  
Bond 2 (A)   = GM:       r_GM  = 0.9 C + eps_GM

In [6]:
b_Viv = np.array([0.4, 0.2, 0.0])
b_GM  = np.array([0.0, 0.0, 0.9])

# Specific risk: Var(r) = b' Sigma_F b + sigma_eps^2 = 1
sig_eps_Viv = np.sqrt(1.0 - float(b_Viv @ FACTOR_CORR @ b_Viv))
sig_eps_GM  = np.sqrt(1.0 - float(b_GM  @ FACTOR_CORR @ b_GM))
print(f"sigma_eps (Vivendi) = {sig_eps_Viv:.4f}")
print(f"sigma_eps (GM)      = {sig_eps_GM:.4f}")

factors_2 = np.random.multivariate_normal([0, 0, 0], FACTOR_CORR, N_SIM)
r_Viv     = factors_2 @ b_Viv + np.random.normal(0, sig_eps_Viv, N_SIM)
r_GM      = factors_2 @ b_GM  + np.random.normal(0, sig_eps_GM,  N_SIM)

bp_Viv = sim_bond_prices(r_Viv, thr_BBB, prices_1)
bp_GM  = sim_bond_prices(r_GM,  thr_A,   prices_2)
port_2 = bp_Viv + bp_GM

var_ex2  = np.sort(port_2)[int(N_SIM * CONF_LEVEL)]
mean_ex2 = np.mean(port_2)
cvar_ex2 = abs(var_ex2 - mean_ex2)

print(f"\nExercise 2  (simulation, N = {N_SIM:,})")
print(f"  muV        = {mean_ex2:.4f}")
print(f"  V*         = {var_ex2:.4f}")
print(f"  Credit VaR = {cvar_ex2:.4f}")

sigma_eps (Vivendi) = 0.8672
sigma_eps (GM)      = 0.4359

Exercise 2  (simulation, N = 100,000)
  muV        = 213.2655
  V*         = 204.3903
  Credit VaR = 8.8752


---
## Exercise 3 — Simulation-based Credit VaR (full portfolio, constant recovery)

All 17 bonds. Recovery = R_mean (constant) for each bond.

In [7]:
RATING_MAP = {1: "AAA", 2: "AA", 3: "A", 4: "BBB",
              5: "BB",  6: "B",  7: "CCC"}

all_prices = []
all_thr    = []
for _, row in bonds.iterrows():
    rn = RATING_MAP[int(row["Rating"])]
    all_prices.append(bond_price_vector(int(row["Maturity"]), row["CR"], row["R_mean"]))
    all_thr.append(z_thresholds(rn))

B_mat   = bonds[["Beta_T", "Beta_L", "Beta_C"]].values
sig_eps = np.array([
    np.sqrt(max(0.0, 1.0 - float(b @ FACTOR_CORR @ b)))
    for b in B_mat
])

# Simulate common factors + idiosyncratic shocks (shared with Exercise 4)
factors_all = np.random.multivariate_normal([0, 0, 0], FACTOR_CORR, N_SIM)
eps_all     = np.column_stack([np.random.normal(0, s, N_SIM) for s in sig_eps])
R_all       = factors_all @ B_mat.T + eps_all   # (N_SIM, 17)

bond_vals_3 = np.column_stack([
    sim_bond_prices(R_all[:, j], all_thr[j], all_prices[j])
    for j in range(len(bonds))
])

port_3   = bond_vals_3.sum(axis=1)
var_ex3  = np.sort(port_3)[int(N_SIM * CONF_LEVEL)]
mean_ex3 = np.mean(port_3)
cvar_ex3 = abs(var_ex3 - mean_ex3)

print(f"Exercise 3  (constant recovery, N = {N_SIM:,})")
print(f"  muV        = {mean_ex3:.4f}")
print(f"  V*         = {var_ex3:.4f}")
print(f"  Credit VaR = {cvar_ex3:.4f}")

Exercise 3  (constant recovery, N = 100,000)
  muV        = 1744.7068
  V*         = 1485.7693
  Credit VaR = 258.9375


---
## Exercise 4 — Simulation-based Credit VaR (full portfolio, stochastic recovery)

Same as Exercise 3 but defaulted bonds draw their recovery from  
Beta(α_j, β_j) matched via method-of-moments to R_mean and R_std.

In [8]:
r_mean = bonds["R_mean"].values / 100
r_std  = bonds["R_std"].values  / 100
common  = r_mean * (1 - r_mean) / r_std**2 - 1
alpha_b = r_mean * common
beta_b  = (1 - r_mean) * common

bond_vals_4 = bond_vals_3.copy()

for j in range(len(bonds)):
    default_price = all_prices[j][-1]
    mask  = bond_vals_4[:, j] == default_price
    n_def = mask.sum()
    if n_def > 0:
        bond_vals_4[mask, j] = beta_dist.rvs(alpha_b[j], beta_b[j], size=n_def) * 100

port_4   = bond_vals_4.sum(axis=1)
var_ex4  = np.sort(port_4)[int(N_SIM * CONF_LEVEL)]
mean_ex4 = np.mean(port_4)
cvar_ex4 = abs(var_ex4 - mean_ex4)

print(f"Exercise 4  (stochastic recovery, N = {N_SIM:,})")
print(f"  muV        = {mean_ex4:.4f}")
print(f"  V*         = {var_ex4:.4f}")
print(f"  Credit VaR = {cvar_ex4:.4f}")

Exercise 4  (stochastic recovery, N = 100,000)
  muV        = 1744.8021
  V*         = 1459.5225
  Credit VaR = 285.2796


## Summary of results

In [9]:
summary = pd.DataFrame({
    "Exercise": [
        "Ex 1 — analytical, rho=0.072",
        "Ex 2 — simulation, 2 bonds",
        "Ex 3 — full portfolio, const recovery",
        "Ex 4 — full portfolio, stoch recovery",
    ],
    "muV":         [mean_ex1, mean_ex2, mean_ex3, mean_ex4],
    "V* (99%)":    [v_star_1, var_ex2,  var_ex3,  var_ex4],
    "Credit VaR":  [cvar_ex1, cvar_ex2, cvar_ex3, cvar_ex4],
}).set_index("Exercise")

print(summary.round(2).to_string())

                                           muV  V* (99%)  Credit VaR
Exercise                                                            
Ex 1 — analytical, rho=0.072            213.27    203.73        9.54
Ex 2 — simulation, 2 bonds              213.27    204.39        8.88
Ex 3 — full portfolio, const recovery  1744.71   1485.77      258.94
Ex 4 — full portfolio, stoch recovery  1744.80   1459.52      285.28
